#### Ensure the notebook path is correct and accessible. If you still get "no active session found" error,
#### try restarting your cluster or re-attaching your notebook to a running cluster.

In [0]:
%run "/Workspace/Users/viggneshwar@gmail.com/databricks/Logistics/Project/Generic_Function/generic_functions_nb"

In [0]:
#%restart_python

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def bonus_calculator(role: str, age: int):
    """
    Calculates bonus percentage based on role and age.
    - Drivers over 50 get 15%
    - Drivers under 30 get 5%
    - Others get 0%
    """
    if role and role.upper() == "DRIVER" and age > 50:
        return 0.15
    elif role and role.upper() == "DRIVER" and age < 30:
        return 0.05
    else:
        return 0.0

# Register as a Spark UDF with explicit return type
bonus_udf = udf(bonus_calculator, DoubleType())

In [0]:
"""
#Example Usage
df = spark.createDataFrame([
    ("DRIVER", 55),
    ("DRIVER", 25),
    ("MANAGER", 40)
], ["role", "age"])

df = df.withColumn("bonus", bonus_udf(df["role"], df["age"]))
df.show()
"""

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def mask_string(name: str):
    """
    Masks a string for privacy:
    - If None, returns None
    - If length <= 2, returns the original string
    - Otherwise, keeps first 2 and last character, masks the rest with '*'
    """
    if name is None:
        return None
    if len(name) <= 2:
        return name
    return name[:2] + "*" * (len(name) - 3) + name[-1]

# Register as a Spark UDF with explicit return type
mask_string_udf = udf(mask_string, StringType())

In [0]:
"""
#Example Usage
df = spark.createDataFrame([("Alice",), ("Bo",), ("C",), (None,)], ["name"])

df_masked = df.withColumn("masked_name", mask_string_udf(df["name"]))
df_masked.show()"""

In [0]:
from pyspark.sql.functions import col, lower, initcap

def staff_data_standardisation_func(df):
    """
    Standardises staff data by:
    - Lowercasing role values
    - Capitalising hub_location values
    - Converting age and shipment_id from words/numbers into integers
    - Renaming first_name, last_name, and hub_location columns
    """
    return (
        df.withColumn("role", lower(col("role")))
          .withColumn("hub_location", initcap(col("hub_location")))
          .withColumn("age", word_to_num_udf(col("age")).cast("int"))
          .withColumn("shipment_id", word_to_num_udf(col("shipment_id")).cast("int"))
          .withColumnRenamed("first_name", "staff_first_name")
          .withColumnRenamed("last_name", "staff_last_name")
          .withColumnRenamed("hub_location", "origin_hub_city")
    )

In [0]:
"""#Example Usage
df = spark.createDataFrame([
    ("Alice", "Smith", "DRIVER", "chennai", "twenty five", "one hundred"),
    ("Bob", "Jones", "Manager", "mumbai", "30", "200")
], ["first_name", "last_name", "role", "hub_location", "age", "shipment_id"])

standardised_df = staff_data_standardisation_func(df)
standardised_df.show()"""

In [0]:
from pyspark.sql.functions import col, lit, upper, to_date, round, current_timestamp

def logistics_shipment_data_standardisation_func(df):
    """
    Standardises logistics shipment data by:
    - Adding domain and ingestion timestamp
    - Setting expedited flag
    - Normalising vehicle type casing
    - Converting shipment_date to proper date format
    - Rounding shipment_cost to 2 decimals
    - Casting shipment_weight_kg to double
    """
    return (
        df.withColumn("domain", lit("Logistics"))
          .withColumn("is_expedited", lit(False).cast("boolean"))
          .withColumn("ingestion_timestamp", current_timestamp())
          .withColumn("vehicle_type", upper(col("vehicle_type")))
          .withColumn("shipment_date", to_date(col("shipment_date"), "yy-MM-dd"))
          .withColumn("shipment_cost", round(col("shipment_cost"), 2))
          .withColumn("shipment_weight_kg", col("shipment_weight_kg").cast("double"))
    )

In [0]:
"""# Example Usage
df = spark.createDataFrame([
    ("truck", "26-01-26", 1234.567, 100.5),
    ("van", "27-01-26", 987.654, 50.2)
], ["vehicle_type", "shipment_date", "shipment_cost", "shipment_weight_kg"])

standardised_df = logistics_shipment_data_standardisation_func(df)
standardised_df.show()"""

In [0]:
from pyspark.sql.functions import col, lit, concat, current_timestamp

def staff_data_enrichedment_func(df):
    """
    Enriches staff data by:
    - Adding load timestamp
    - Creating full_name from first and last names
    - Selecting relevant columns for downstream use
    """
    return (
        df.withColumn("load_dt", current_timestamp())
          .withColumn("full_name", concat(col("staff_first_name"), lit(" "), col("staff_last_name")))
          .select(
              "shipment_id",
              "full_name",
              "age",
              "role",
              "origin_hub_city",
              "vehicle_type",
              "load_dt",
              "data_source"
          )
    )

In [0]:
"""#Example Usage
df = spark.createDataFrame([
    (100, "Alice", "Smith", 25, "driver", "Chennai", "truck", "systemA"),
    (200, "Bob", "Jones", 30, "manager", "Mumbai", "van", "systemB")
], ["shipment_id", "staff_first_name", "staff_last_name", "age", "role", "origin_hub_city", "vehicle_type", "data_source"])

enriched_df = staff_data_enrichedment_func(df)
enriched_df.show()"""

In [0]:
from pyspark.sql.functions import (
    col, lit, concat, year, month, dayofweek, dayofmonth,
    when, round, current_date, datediff, substring, length, upper, to_date
)

def logistics_shipment_data_enrichment_func(df):
    """
    Enriches logistics shipment data by adding derived fields:
    - Route segment and lane
    - Vehicle identifier
    - Shipment year/month/day
    - Weekend flag
    - Expedited flag based on status
    - Cost per kg (safe division)
    - Days since shipment
    - Tax amount (18%)
    - Order prefix and sequence
    """
    return (
        df.withColumn("route_segment", concat(col("source_city"), lit("-"), col("destination_city")))
          .withColumn("vehicle_identifier", concat(col("vehicle_type"), lit("_"), col("shipment_id")))
          .withColumn("shipment_year", year(col("shipment_date")))
          .withColumn("shipment_month", month(col("shipment_date")))
          .withColumn("shipment_day", dayofmonth(col("shipment_date")))
          .withColumn("is_weekend", when((dayofweek(col("shipment_date")) == 1) | (dayofweek(col("shipment_date")) == 7), True).otherwise(False))
          .withColumn("is_expedited", when((col("shipment_status").isin("IN_TRANSIT", "DELIVERED")), True).otherwise(False))
          .withColumn("cost_per_kg", when(col("shipment_weight_kg") != 0, col("shipment_cost") / col("shipment_weight_kg")).otherwise(None))
          .withColumn("days_since_shipment", datediff(current_date(), col("shipment_date")))
          .withColumn("tax_amount", col("shipment_cost") * 0.18)
          .withColumn("order_prefix", substring(col("order_id"), 1, 3))
          .withColumn("order_sequence", substring(col("order_id"), 4, length(col("order_id"))))
          .withColumn("route_lane", concat(col("source_city"), lit("->"), col("destination_city")))
    )

In [0]:
"""#Example Usage
df = spark.createDataFrame([
    ("Chennai", "Mumbai", "truck", 101, "2026-02-01", 1200.0, 100.0, "IN_TRANSIT", "ORD12345"),
    ("Delhi", "Bangalore", "van", 102, "2026-01-28", 800.0, 50.0, "PENDING", "ORD67890")
], ["source_city", "destination_city", "vehicle_type", "shipment_id", "shipment_date", "shipment_cost", "shipment_weight_kg", "shipment_status", "order_id"])

enriched_df = logistics_shipment_data_enrichment_func(df)
enriched_df.show()"""

In [0]:
from pyspark.sql.functions import col

def staff_data_customize_func(df):
    """
    Customises staff data by:
    - Adding projected bonus based on role and age
    - Masking full_name for privacy
    """
    return (
        df.withColumn("projected_bonus", bonus_udf(col("role"), col("age")))
          .withColumn("full_name", mask_string_udf(col("full_name")))
    )

In [0]:
"""#Example Usage
df = spark.createDataFrame([
    (100, "Alice Smith", 55, "driver"),
    (200, "Bob Jones", 25, "driver"),
    (300, "Charlie Brown", 40, "manager")
], ["shipment_id", "full_name", "age", "role"])

customised_df = staff_data_customize_func(df)
customised_df.show()"""

In [0]:
#silver_db = 'logistics_proj.shipment_logistics_data'
#spark.sql(f"SELECT * FROM {silver_db}.logistics_shipment_silver_tbl")

In [0]:
"""#Example Usage
df_silver = spark.sql(f"SELECT * FROM {silver_db}.logistics_shipment_silver_tbl")
df_silver.show(5)"""

In [0]:
from pyspark.sql.functions import col, upper, concat, lit, when

def logistics_shipment_gold_curation(table: str):
    """
    Curates the gold layer for logistics shipment data by:
    - Renaming and standardising key fields
    - Adding INR formatted cost
    - Deriving high-value shipment flag
    """
    return spark.sql(f"""
        SELECT
            shipment_id AS log_shipment_id,
            order_id,
            UPPER(source_city) AS source_city,
            UPPER(destination_city) AS destination_city,
            shipment_status,
            cargo_type,
            vehicle_type AS shipment_vehicle_type,
            payment_mode,
            shipment_weight_kg,
            shipment_cost,
            CONCAT('₹', CAST(shipment_cost AS STRING)) AS shipment_cost_inr,
            shipment_date,
            domain,
            ingestion_timestamp,
            is_expedited,
            route_segment,
            vehicle_identifier,
            shipment_year,
            shipment_month,
            is_weekend,
            cost_per_kg,
            days_since_shipment,
            tax_amount,
            order_prefix,
            order_sequence,
            --shipment_year,
            --shipment_month,
            shipment_day,
            route_lane,
            CASE WHEN shipment_cost > 50000 THEN TRUE ELSE FALSE END AS is_high_value
        FROM {table}
    """)

In [0]:
"""#Example Usage
gold_df = logistics_shipment_gold_curation("logistics_proj.shipment_logistics_data.logistics_shipment_silver_tbl")
gold_df.show(5)"""